## Agentic RAG and tool-augmented reasoning

This third study in the RAG series tests whether a controller should retrieve again, choose another search tool, or stop. Fixed retrieval is a serious comparator. All reported values come from retained experiment traces; no-retrieval and oracle-document controls help distinguish retrieval gains from remaining context and generation limits.

In [ ]:
from pathlib import Path
import sys, json, inspect, hashlib
ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
if ROOT.name!='Agentic RAG': ROOT=ROOT/'Agentic RAG'
sys.path.insert(0,str(ROOT/'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
get_ipython().run_line_magic('matplotlib','inline')
from agentic_rag.artifacts import read,sha,digest,fingerprint
from agentic_rag.evaluation import score,aggregate,paired_interval
OUT=ROOT/'reports'
(OUT/'figures').mkdir(parents=True,exist_ok=True)
metrics=read(OUT/'metrics.json')
reliability=read(OUT/'reliability.json')
intervals=read(OUT/'paired-intervals.json')
config=read(ROOT/'configs/study.json')
freeze=read(OUT/'selection-freeze.json')
names=['hotpot','squad']
strategies=['fixed','adaptive','plan','agent','verify','none','oracle']
def records(label,name,split='evaluation'):
    manifest=read(ROOT/f'data/manifests/{label}-{name}-{split}.json')
    return [read(ROOT/e['path']) for e in manifest['rows']]
def scores(label,name,split='evaluation'):
    return read(ROOT/f'data/scores/{label}-{name}-{split}.json')
def frame(labels,names=names,split='evaluation'):
    return pd.DataFrame([dict(dataset=n,strategy=s,**metrics[f'{s}-{n}-{split}']) for n in names for s in labels])
plt.rcParams.update({'figure.figsize':(10,4),'axes.spines.top':False,'axes.spines.right':False})
display(pd.DataFrame(config.items(),columns=['setting','value']))

## Research questions and comparison logic

The primary question is whether dynamic evidence gathering improves answer F1. Secondary questions concern evidence recall, tool selection, plan quality, verification-induced abstention, unnecessary retrieval and inference costs. The comparisons change control logic while fixing the main answer model. They do not assume that more steps help.

In [ ]:
display(frame(strategies)[['dataset','strategy','n','groups']])

## Related methods and design boundaries

[ReAct](https://arxiv.org/abs/2210.03629) motivates interacting with evidence tools; [IRCoT](https://aclanthology.org/2023.acl-long.557/) motivates evidence-dependent queries. [Adaptive-RAG](https://aclanthology.org/2024.naacl-long.389/) studies complexity-based routing, whereas this rule baseline checks source diversity and lexical coverage. [Self-RAG](https://arxiv.org/abs/2310.11511) learns reflection tokens; our external NLI gate is a different mechanism. This study does not reproduce those papers' training procedures or benchmark results.

In [ ]:
display(pd.DataFrame([['Fixed','No control-model overhead'],['Rules','Deterministic sufficiency proxy'],['Plan','Two actions before observations'],['Agent','Observe and choose again'],['Verify','External generated-claim check']],columns=['strategy','design choice']))

## Reference findings and preservation

Project 1's fusion gains and reranking losses motivate controlled tests. Project 2 found that practical FLAN-T5-base could outperform structured Qwen generation. Its citation and NLI limitations motivate separate source-label and claim-support diagnostics here. The prior files remain unchanged.

In [ ]:
preserved=read(OUT/'preservation.json')
assert all(sha(ROOT.parent/p)==h for p,h in preserved.items())
print('Preserved prior-project files:',len(preserved))
print('Cleanup:',read(OUT/'cleanup.json'))

## Hardware and executable environment

The environment audit measures the actual host. Available memory changes during the run. Local CUDA overlays and model snapshots are reused without modifying the earlier projects. No paid API is used.

In [ ]:
environment=read(OUT/'environment.json')
display(pd.DataFrame([(k,v) for k,v in environment.items() if k not in ['packages','reusable']],columns=['item','observed']))
display(pd.DataFrame(environment['reusable']))
print(read(OUT/'runtime-environment.json')['actual_imports'])

## Dataset provenance and alternatives

HotpotQA distractor is the primary multihop benchmark. SQuAD 2.0 contributes genuine unanswerable questions. All previous SQuAD development articles had been touched, so the fresh complementary study uses article-disjoint partitions of the public training release. MuSiQue and 2Wiki were reviewed but not acquired. These local partitions are not official test results.

In [ ]:
sources=read(ROOT/'sources.lock.json')
for source in sources:
    path=ROOT/('data/raw/squad-train.json' if source['name']=='squad' else 'data/raw/hotpot.json')
    assert sha(path)==source['sha256']
display(pd.DataFrame(sources)[['name','url','bytes','sha256','license']])

## Partitions and prior exposure

Group allocation precedes question sampling. Development selects the reference; validation is a diagnostic check; evaluation stays untouched until the freeze. Prior question IDs are excluded and SQuAD is grouped by article. HotpotQA can still share pages across questions. The published sampling record can reconstruct exact partitions on a fresh checkout.

In [ ]:
validation=read(OUT/'data-validation.json')
partitions={n:read(ROOT/f'data/partitions/{n}.json') for n in names}
split_rows=[]
for name in names:
    for split,values in validation[name]['partitions'].items():
        assert digest(partitions[name][split])==values['hash']
        split_rows.append(dict(dataset=name,split=split,**{k:v for k,v in values.items() if k!='ids'}))
display(pd.DataFrame(split_rows))

## Question types and answerability

HotpotQA bridge and comparison questions pose different evidence needs. SQuAD class prevalence matters because abstention alone can score well. The class counts below retain the original labels.

In [ ]:
eda=pd.DataFrame([dict(dataset=n,split=s,kind=r['kind'],words=len(r['question'].split()),documents=len(r['documents'])) for n in names for s,rows in partitions[n].items() for r in rows])
display(eda.groupby(['dataset','split','kind']).size().rename('count').to_frame())

## Exploratory question distribution

Question length and candidate count characterize this bounded study. SQuAD supplies one paragraph per question, so selecting another retrieval backend does not create another document of evidence.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
for name,group in eda.groupby('dataset'):
    axes[0].hist(group.words,bins=20,alpha=.5,label=name)
    axes[1].hist(group.documents,bins=10,alpha=.5,label=name)
axes[0].set_xlabel('Question words'); axes[1].set_xlabel('Candidate documents')
for ax in axes: ax.legend()
plt.tight_layout(); fig.savefig(OUT/'figures/question-distribution.png',dpi=160,bbox_inches='tight'); plt.show()

## Annotation validation and duplicate checks

Original labels are retained. Out-of-range supporting facts are reported and omitted only from the sentence-recall denominator. Question and candidate-body duplicates are counted, and split separation is checked. HotpotQA groups questions, so document reuse across partitions is measured rather than hidden. SQuAD answer offsets are validated against original paragraphs during loading.

In [ ]:
display(pd.DataFrame([dict(dataset=n,original_questions=v['original_questions'],previous_questions_excluded=v['previous_questions_excluded'],available_questions=v['available_questions'],duplicate_questions=v['duplicate_question_texts'],invalid_facts=len(v['invalid_facts'])) for n,v in validation.items()]))
display(pd.DataFrame(read(OUT/'data-structure.json')).T)
for name in names:
    sets=[{r['group'] for r in rows} for rows in partitions[name].values()]
    assert all(not sets[i]&sets[j] for i in range(3) for j in range(i))

## Structured agent state and action validation

The controller stores evidence, steps, call counts, tokens, failures and stage times. JSON actions can only dispatch registered read-only tools. Malformed outputs, unknown tools, repeated actions and budget exhaustion are handled in Python rather than delegated to instruction following.

In [ ]:
from agentic_rag.controller import Action,State,Controller,SEARCH,LOOKUP
print(inspect.getsource(Action))
print(inspect.getsource(State))

## Evidence store and passage provenance

Passages retain original titles, sentence offsets and deterministic IDs. Deduplication operates on passage IDs. Source titles identify documents within a question; they are not globally unique corpus identifiers.

In [ ]:
from agentic_rag.tools import Index,EvidenceStore
from agentic_rag.data import task
example=partitions['hotpot']['evaluation'][0]
index=Index(task(example))
display(pd.DataFrame([dict(id=p.id,document=p.document,sentence_indices=p.sentence_indices,words=len(p.text.split())) for p in index.passages]).head(12))
print(inspect.getsource(EvidenceStore))

## Search tools and ranking formulas

BM25 uses positive Robertson IDF with k1=1.5 and b=0.75. Dense search uses normalized BGE-small embeddings. Hybrid search fuses sparse and dense ranks with constant 60, and MiniLM reranks up to 12 fused candidates. Fetch and title-link lookup use exact source IDs.

In [ ]:
from agentic_rag.tools import BM25Okapi
print(inspect.getsource(BM25Okapi))
print(inspect.getsource(Index.execute))

## Graph navigation and evidence expansion

Edges are deterministic title mentions in source passages. They support navigation without claiming that a semantic relationship is verified. Expansion returns bounded passages around the best query-overlap anchor inside a chosen document. It introduces no gold evidence labels.

In [ ]:
print('Example title-mention links:',{k:v for k,v in index.links.items() if v})
graph_stats=[]
for row in partitions['hotpot']['evaluation']:
    graph=Index(task(row)).links
    graph_stats.append(dict(nodes=len(graph),edges=sum(map(len,graph.values())),has_edge=any(graph.values())))
display(pd.DataFrame(graph_stats).describe())
print('Graph construction uses source text only:',inspect.getsource(Index.__init__))

## Five distinct control strategies

Fixed RAG retrieves once. Rules expand when document diversity or question-token coverage is low. Plan-and-execute commits to up to two actions before reading evidence. The reactive agent observes tool output before choosing again. Verification-guided control tests generated claims, permits further gathering and can abstain.

In [ ]:
print(inspect.getsource(Controller.run))

## Model snapshots and inference formats

Each snapshot is pinned by revision and file hash. FLAN-T5-base answers all primary conditions. Planner candidates include FLAN and two Qwen sizes. BGE, MiniLM reranking and DeBERTa NLI serve separate roles. Greedy Qwen decoding is an experimental choice, different from its recommended sampling setup.

In [ ]:
models=read(ROOT/'models.lock.json')
display(pd.DataFrame([dict(alias=a,model=r['id'],revision=r['revision'],license=r['license'],bytes=r['bytes']) for a,r in models.items()]))
print(read(OUT/'resources.json'))

## Development-only planner comparison

The initial pilot exposed wrong lookup fields and repeated actions. Before held-out evaluation, the prompt gained candidate titles and explicit schemas, no-source outputs were gated, and planner utility was revised to include evidence usefulness. A contradictory single-object example in the plan-array prompt was also corrected and checked on eight development questions. Earlier revisions remain local. The final model pilot uses eight development questions per dataset and cannot establish a general model ranking.

In [ ]:
planner=read(OUT/'planner-selection.json')
print('Criterion:',planner['criterion']); print('Selected:',planner['selected'])
display(pd.DataFrame([dict(model=a,dataset=n,**v) for a,ds in planner['results'].items() for n,v in ds.items()])[['model','dataset','n','f1','evidence_recall','invalid_actions','calls','p50_ms']])
display(pd.DataFrame(read(OUT/'plan-schema-probe.json')).T[['n','calls','invalid_actions','f1']])

## Exploratory answer-model comparison

The same fixed retrieval backend is used for the small answer-model check. Tokenizers and input ceilings differ, and the shared context setting can select different passages. FLAN remains the prespecified primary generator; this is an exploratory system comparison.

In [ ]:
comparison=read(OUT/'generator-comparison.json')
print(comparison['policy'])
display(pd.DataFrame([dict(model=a,dataset=n,**v) for a,ds in comparison['results'].items() for n,v in ds.items()])[['model','dataset','n','f1','answered','model_support','p50_ms']])

## Retrieval selection on development data

Four retrieval backends compete on 32 development questions per dataset with the answer model fixed. The pooled F1 winner becomes the fixed reference. Development choices never use evaluation answers.

In [ ]:
display(frame(['retrieval_bm25','retrieval_dense','retrieval_hybrid','retrieval_rerank'],split='development')[['dataset','strategy','n','f1','evidence_recall','p50_ms']])
print('Selected retrieval:',freeze['retrieval'])

## Validation and protocol freeze

Validation measures the frozen design without tuning thresholds. The freeze precedes every final-evaluation checkpoint. All primary conditions share the source fingerprint; raw generations record their actual prompts and settings.

In [ ]:
display(frame(['fixed','adaptive','plan','agent','verify'],split='validation')[['dataset','strategy','f1','answered','calls','invalid_actions']])
assert freeze['source_identity']==fingerprint()
print('Freeze:',freeze['timestamp_utc'],freeze['freeze_hash'])

## Main answer comparison

The full held-out comparison uses 128 questions per dataset. EM measures exact normalized answers; token F1 gives partial credit. Performance can worsen under extra planning or verification, and no method is assumed to dominate.

In [ ]:
main=frame(strategies)
display(main[['dataset','strategy','n','em','f1','answered','selective_em']])

## Bridge, comparison and answerability slices

The original question-type labels identify where aggregate gains or losses occur. HotpotQA bridge and comparison questions have different evidence needs. For SQuAD, separate answerable and unanswerable rows distinguish answer quality from class-prior effects. These smaller slices are descriptive and should be read with their sample sizes.

In [ ]:
type_metrics=read(OUT/'question-type-metrics.json')
display(pd.DataFrame([dict(dataset=n,strategy=s,kind=k,**v) for n in names for s in ['fixed','adaptive','plan','agent','verify'] for k,v in type_metrics[f'{s}-{n}-evaluation'].items()])[['dataset','strategy','kind','n','em','f1','answered','evidence_recall']])

## Answer quality figure

The bars show point estimates on matched question sets. Paired intervals below describe uncertainty in differences. The oracle-document row still has a finite context and the same generator.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,name in zip(axes,names):
    main[main.dataset==name].set_index('strategy')[['em','f1']].plot.bar(ax=ax,ylim=(0,1),title=name)
plt.tight_layout(); fig.savefig(OUT/'figures/answer-quality.png',dpi=160,bbox_inches='tight'); plt.show()

## Evidence-document and supporting-fact recall

Packed-context recall measures original gold titles retained with the final answer result, rather than every source ever returned. Separate returned-evidence and memory recall reveal retrieval gains lost through deduplication, context limits or final budget abstention. HotpotQA sentence recall requires complete original sentence containment.

In [ ]:
display(main[['dataset','strategy','evidence_recall','supporting_fact_recall','citation_recall']])
display(pd.DataFrame([dict(dataset=n,strategy=s,returned_recall=reliability[f'{s}-{n}-evaluation']['returned_evidence_recall'],memory_recall=reliability[f'{s}-{n}-evaluation']['memory_evidence_recall']) for n in names for s in ['fixed','adaptive','plan','agent','verify']]))

## Evidence quality figure

The HotpotQA document and sentence diagnostics show whether additional control finds useful evidence before generation. SQuAD's single paragraph makes document recall a limited discriminator.

In [ ]:
main[main.dataset=='hotpot'].set_index('strategy')[['evidence_recall','supporting_fact_recall']].plot.bar(ylim=(0,1))
plt.ylabel('Recall in packed context'); plt.tight_layout(); plt.gcf().savefig(OUT/'figures/evidence-recall.png',dpi=160,bbox_inches='tight'); plt.show()

## Citation validity and source-label agreement

Citation assignment is a post hoc overlap heuristic. The validity flag describes the proposal before final abstention: its IDs must occur in packed context. The separate uncited-answer count checks final accepted outputs. Precision and recall compare original source labels. These checks do not establish model reliance or entailment, and one lexical citation can omit another document needed for a multihop answer.

In [ ]:
display(main[['dataset','strategy','citation_precision','citation_recall','citation_valid']])
display(pd.DataFrame([dict(dataset=n,strategy=s,uncited_answers=reliability[f'{s}-{n}-evaluation']['answer_without_valid_citation']) for n in names for s in strategies]))

## Actual generated claims and model support

A separate FLAN call rewrites each proposal as a declarative claim. DeBERTa scores it against the cited passage. The controller score defaults to zero for unscored proposals; conditional averages below use retained final claims only. Intermediate NLI probabilities are not logged, although generation calls and cumulative verifier time are preserved. Claim rewriting, truncation and question mismatch limit these model estimates; they remain separate from evidence annotations.

In [ ]:
display(main[['dataset','strategy','model_support','model_contradiction','claim_contains_answer']])
display(pd.DataFrame([dict(dataset=n,strategy=s,claim_proposals=reliability[f'{s}-{n}-evaluation']['claim_proposals'],mean_scored_entailment=reliability[f'{s}-{n}-evaluation']['mean_nli_entailment_on_claims'],accepted_claim_entailment=reliability[f'{s}-{n}-evaluation']['mean_nli_entailment_on_accepted_claims']) for n in names for s in ['fixed','adaptive','plan','agent','verify']]))
print(inspect.getsource(Controller.answer))

## Missing and conflicting evidence diagnostics

Missing annotated documents can be counted, but a missing label does not prove the answer is unsupported. The controller checks NLI only against its assigned citation. This separate post-run replay checks the claim against every packed passage and flags simultaneous entailment and contradiction signals. The frozen controller never sees these extra checks. They reveal possible missed conflicts for review and are not independently verified contradictions.

In [ ]:
conflicts=read(OUT/'conflict-diagnostics.json')
print(conflicts['scope'])
display(pd.DataFrame(conflicts['results']).T)
print('Replay time, separate from benchmark latency (ms):',conflicts['replay_ms'])

## Citation-window case review

The assistant review inspects real proposed answers, generated claims and their assigned source windows. It selects two nonempty proposals for each dataset and strategy, excludes previously selected questions, and prefers rejected verification proposals. The review is unblinded to reference answers and has no independent human rater. Its counts describe these cases, not a population faithfulness estimate. Hashes prevent reuse of labels after answers or evidence change.

In [ ]:
review=read(OUT/'support-review.json')
print(review['annotator']); print(review['selection']); print(review['criteria'])
print('Unique questions:',review['unique_questions']); print('All reviewed proposals:',review['counts']); print('Accepted proposals:',review['accepted_counts'])
display(pd.DataFrame(review['cases']))

## Verification and abstention

The verification agent can reject an answer-string match or accept an irrelevant entailed claim. An answer-string match is not by itself evidence of support, and an EM mismatch can miss valid aliases. Coverage and selective EM expose the acceptance tradeoff. SQuAD's original unanswerable cases allow a false-answer rate; always-abstain helps identify prevalence-driven gains.

In [ ]:
display(main[['dataset','strategy','f1','answered','selective_em','false_answer_rate']])
display(pd.DataFrame([dict(dataset=n,**read(OUT/f'always-abstain-{n}.json')) for n in names]))
display(pd.DataFrame([dict(dataset=n,strategy=s,high_support_em_mismatch=reliability[f'{s}-{n}-evaluation']['high_support_answer_em_mismatch'],rejected_proposals=reliability[f'{s}-{n}-evaluation']['rejected_proposals'],rejected_exact_matches=reliability[f'{s}-{n}-evaluation']['rejected_exact_match_proposals']) for n in names for s in ['fixed','agent','verify']]))

## Risk and coverage

Selective EM is descriptive of the accepted subset rather than accuracy on all questions. It should be read with the number of accepted answers and the full-set F1.

In [ ]:
fig,ax=plt.subplots()
for name in names:
    group=main[(main.dataset==name)&main.strategy.isin(['fixed','adaptive','plan','agent','verify'])]
    ax.scatter(group.answered,group.selective_em,label=name)
    for _,r in group.iterrows(): ax.annotate(r.strategy,(r.answered,r.selective_em))
ax.set(xlabel='Answer coverage',ylabel='Selective exact match',xlim=(0,1),ylim=(0,1)); ax.legend(); fig.savefig(OUT/'figures/risk-coverage.png',dpi=160,bbox_inches='tight'); plt.show()

## Paired uncertainty

Bootstrap resampling keeps answers paired and resamples SQuAD articles as clusters. Intervals use 2,000 replicates, are pointwise and are not adjusted for multiple comparisons. Small cluster counts can leave wide intervals.

In [ ]:
paired=pd.DataFrame([dict(configuration=k,**v) for k,v in intervals.items() if any(k==f'{s}-{n}-evaluation' for s in ['adaptive','plan','agent','verify','none','oracle'] for n in names)])
display(paired)

## Matched difference figure

Differences are relative to fixed RAG on the same 128 questions. The figure shows the direction and uncertainty of each strategy's answer-F1 change.

In [ ]:
fig,ax=plt.subplots(figsize=(10,6))
x=np.arange(len(paired)); mid=paired.delta.to_numpy(); low=paired.low.to_numpy(); high=paired.high.to_numpy()
ax.errorbar(mid,x,xerr=np.vstack([mid-low,high-mid]),fmt='o'); ax.axvline(0,color='grey',linestyle='--')
ax.set_yticks(x,paired.configuration); ax.set_xlabel('F1 difference from fixed'); plt.tight_layout(); fig.savefig(OUT/'figures/paired-f1.png',dpi=160,bbox_inches='tight'); plt.show()

## Search budgets, context and evidence memory

The ablations change one setting at a time on the same first 32 evaluation questions. A one-step agent still spends planning inference; it is not the fixed pipeline. Deduplication affects packed context as well as storage.

In [ ]:
budget_labels=['steps1','calls2','context280','no_dedup']
display(frame(budget_labels)[['dataset','strategy','n','f1','evidence_recall','calls','tokens','p50_ms']])

## Registry and stopping ablations

Search-only and BM25-only registries test whether exposing more tools helps. Verification thresholds 0 and 0.8 change acceptance and stopping relative to the prespecified 0.5 rule. These held-out sensitivity results do not select another final model.

In [ ]:
display(frame(['search_only','bm25_only','verify_zero','verify_high'])[['dataset','strategy','n','f1','answered','calls','invalid_actions']])

## Bad-tool outputs and recovery

The first actual call can return nothing, passages from the tail of the candidate order, or a synthetic instruction attack. Tail-order passages are an irrelevant-result proxy and may coincidentally contain gold evidence. Each perturbation uses the same underlying question set; comparisons with an unperturbed subset are matched.

In [ ]:
display(frame(['empty_first','irrelevant_first','injection_first'])[['dataset','strategy','n','f1','evidence_recall','additional_evidence_gain','calls','invalid_actions']])
display(pd.DataFrame([dict(configuration=k,**v) for k,v in intervals.items() if any(k.startswith(x) for x in ['empty_first','irrelevant_first','injection_first'])]))

## Tool choices and reliability

Successful dispatch can return empty evidence. Invalid-action counts include rejected schema fields, repeated actions and invalid source IDs; they are distinct from backend exceptions. Safe budget termination is not a runaway loop, though it can stop too early.

In [ ]:
display(main[['dataset','strategy','calls','steps','tool_success_rate','invalid_action_rate','empty_calls','budget_terminated']])
display(pd.DataFrame([dict(dataset=n,strategy=s,**reliability[f'{s}-{n}-evaluation']['tool_distribution']) for n in names for s in ['plan','agent','verify']]).fillna(0))

## Unnecessary calls and retrieval gains

Duplicate-evidence frequency is a conservative waste proxy. The matched one-step contrast also counts extra calls with identical answers or no F1 gain. Recovery means a later call retrieved an annotated document after the first-result intervention. These diagnostics are not independent causal-necessity labels.

In [ ]:
display(main[['dataset','strategy','calls','redundant_calls','additional_evidence_gain']])
display(pd.DataFrame(read(OUT/'call-utility.json')).T)
display(pd.DataFrame(read(OUT/'retrieval-recovery.json')).T)
display(pd.DataFrame([dict(dataset=n,strategy=s,reasons=reliability[f'{s}-{n}-evaluation']['controller_stop_reasons']) for n in names for s in strategies]))

## A real multi-step trace

The example is selected from actual HotpotQA agent traces by executed call count, without changing the outcome. The trace retains returned passages and failed actions so that evidence lookups can be inspected. This is an illustrative post hoc case, not a representative estimate.

In [ ]:
agent_traces=records('agent','hotpot')
trace=max(agent_traces,key=lambda r:r['state']['calls'])
row=next(r for r in partitions['hotpot']['evaluation'] if r['id']==trace['id'])
print('Question:',row['question']); print('Reference:',row['answers']); print('Answer:',trace['answer']); print('Stop:',trace['reason'])
display(pd.DataFrame(trace['state']['trace']))
print('Plan model outputs:',[g['text'] for g in trace['state']['generations'] if g['stage']=='planning'])

## Where agentic control harms answers

This matched case selects the largest fixed-to-agent F1 loss in the recorded HotpotQA evaluation. Better evidence recall is not sufficient if planning, context packing or generation discards the useful fact. Source windows and actual outputs remain visible for diagnosis.

In [ ]:
fixed_scores=scores('fixed','hotpot'); agent_scores=scores('agent','hotpot')
loss=max(range(len(fixed_scores)),key=lambda i:fixed_scores[i]['f1']-agent_scores[i]['f1'])
fixed_trace=records('fixed','hotpot')[loss]; agent_trace=agent_traces[loss]
row=partitions['hotpot']['evaluation'][loss]
print(row['question'],row['answers']); print('Fixed:',fixed_trace['answer']); print('Agent:',agent_trace['answer'])
display(pd.DataFrame(agent_trace['context'])); display(pd.DataFrame(agent_trace['state']['trace']))

## Verification rejection example

The first held-out rejected proposal illustrates a gate decision. It does not by itself show that the gate was correct. Inspect the generated claim, source text, answer annotation and NLI score together.

In [ ]:
rejected=[r for r in records('verify','hotpot') if r.get('rejected_answer')]
if rejected:
    case=rejected[0]; row=next(r for r in partitions['hotpot']['evaluation'] if r['id']==case['id'])
    print(row['question'],row['answers']); print('Rejected:',case['rejected_answer']); print('Claim:',case['claim']); print('NLI:',case['support'])
    display(pd.DataFrame(case['context']))
else: print('No nonempty rejected proposal in this recorded condition.')

## Latency decomposition

Times include actual inference and tool work. The fixed comparator receives the same claim/NLI diagnostics. Per-question end-to-end time includes indexing; model loading is separate. Checkpoint reload speed is not reported as inference latency.

In [ ]:
display(main[['dataset','strategy','planning_ms','retrieval_ms','verification_ms','generation_ms','p50_ms','p95_ms','tokens']])

## Runtime cost figure

Mean stage costs reveal where orchestration adds work. These are sequential, warm-run observations with uncontrolled background activity; they are not throughput estimates for a serving system.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
for ax,name in zip(axes,names):
    stage=main[(main.dataset==name)&main.strategy.isin(['fixed','adaptive','plan','agent','verify'])].set_index('strategy')
    (stage[['planning_ms','retrieval_ms','verification_ms','generation_ms']]/1000).plot.bar(stacked=True,ax=ax,title=name)
    ax.set_ylabel('Mean seconds')
plt.tight_layout(); fig.savefig(OUT/'figures/stage-costs.png',dpi=160,bbox_inches='tight'); plt.show()

## Memory and loading costs

Process peak working set and CUDA allocated/reserved peaks cover whole sequential processes and co-resident models. They do not isolate a single method's memory. Stage-specific resource files preserve observations across separate research commands.

In [ ]:
resources=[dict(stage=p.stem,**read(p)) for p in (OUT/'resource-stages').glob('*.json')]
resource_frame=pd.DataFrame(resources)
display(resource_frame[['stage','device','rss_bytes','peak_working_set_bytes','cuda_peak_allocated_bytes','cuda_peak_reserved_bytes','generator_load_ms']])
print('Maximum observed CUDA allocation GiB:',resource_frame.cuda_peak_allocated_bytes.max()/2**30)

## Experiment-to-report consistency

Replaying the scorer on retained traces protects aggregate tables from transcription errors. Row hashes verify the annotated input. A separate standard-library implementation checks answer EM/F1 against each dataset's metric conventions. Neither check certifies model correctness or replaces independent evidence annotation.

In [ ]:
checked=0
for name in names:
    annotations={r['id']:r for r in partitions[name]['evaluation']}
    for strategy in strategies:
        rows=[score(r,annotations[r['id']]) for r in records(strategy,name)]
        assert aggregate(rows)==metrics[f'{strategy}-{name}-evaluation']
        checked+=len(rows)
print('Primary question results recomputed from trace files:',checked)
print('Separate answer-metric implementation:',read(OUT/'metric-convention-audit.json'))

## Tests, security scope and recovery

The pre-publication tests cover action validation, finite transitions, budgets, dispatch, deduplication, citations, deterministic behavior, malformed outputs, unavailable services and injected instructions. The final suite also checks the public notebook after its creation. Separate uncached model probes exercise five workflows twice and check simple NLI pairs; a real CLI smoke test checks the command path. These synthetic probes are not benchmark or comprehensive security evidence.

In [ ]:
import xml.etree.ElementTree as ET
test_root=ET.parse(OUT/'tests-notebook.xml').getroot()
display(pd.DataFrame([s.attrib for s in test_root.iter('testsuite')]))
probes=read(OUT/'workflow-probes.json')
print(probes['scope']); display(pd.DataFrame(probes['results'])[['strategy','answer','repeat_signature_equal','calls','tokens']])
print('NLI label map:',probes['nli_labels']); display(pd.DataFrame(probes['nli_semantic_probes']))
print('CLI smoke:',read(OUT/'cli-smoke.json'))
print('Retained failed/interrupt records:',read(OUT/'execution-ledger.json')['failed_records'])

## Reproduce and resume

The scripts acquire and verify data/models, reconstruct the published sampling lock, run staged experiments, rebuild reports, execute this notebook in a clean kernel, and create its output-free public copy. Completed unchanged question runs are reused. Source/config/model changes create new cache identities. Never replace the local executed notebook to regenerate the public copy.

In [ ]:
print('Source identity:',fingerprint())
print('Freeze source identity:',freeze['source_identity'])
print('Completed stage ledger:',read(OUT/'execution-ledger.json'))
assert read(OUT/'experiment-provenance.json')['source_identity']==fingerprint()

## Conclusions and limits

On hotpot, fixed RAG scored 0.493 F1 and the reactive agent scored 0.431. The paired difference was -0.062, with a 95% interval of [-0.132, +0.014]. Median agent latency was 9.3 times the fixed reference.

On squad, fixed RAG scored 0.770 F1 and the reactive agent scored 0.748. The paired difference was -0.022, with a 95% interval of [-0.048, -0.001]. Median agent latency was 10.0 times the fixed reference.

An increase in source recall can coexist with worse answers, and abstention can improve SQuAD scores through class prevalence. These bounded, single-host results cannot establish open-domain quality, general reasoning ability, independent faithfulness or production security. Larger planners, external multihop replication, independent evidence judgments and repeated timing measurements remain unfinished.

In [ ]:
conclusion=main[['dataset','strategy','f1','evidence_recall','answered','p50_ms']].copy()
display(conclusion)
print('Unfinished work:',read(OUT/'execution-ledger.json')['unfinished'])